# PACSUN2603 Data
This notebook unifies everything into one dataframe for each experiment to make plotting easier later.

In [10]:
import pandas as pd
import pytz
from pvlib.solarposition import get_solarposition
import time

## Geographic Data

In [11]:
gdf = pd.read_csv("ISAC_PACSUN2603.csv", comment='#')
gdf.columns

Index(['DataId', 'CommId', 'DeviceName', 'DeviceDateTime', 'AgeInSeconds',
       'BatteryVoltage', 'GpsQuality', 'Latitude', 'Longitude',
       'SubmergedBoolean', 'Temperature0cm', 'Unnamed: 11', 'station'],
      dtype='str')

In [12]:
# Read the CSV data directly from the file.
# The 'comment='#' will automatically skip lines starting with '#'.
gdf = pd.read_csv("ISAC_PACSUN2603.csv", comment='#')

# Convert 'iso_time' column to datetime objects
# gdf['Datetime'] = pd.to_datetime(gdf['DeviceDateTime'], unit = 's').dt.tz_localize('Pacific/Tongatapu')
gdf['Datetime'] = pd.to_datetime(gdf['DeviceDateTime'], unit = 's')
gdf['unixTime'] = (gdf['Datetime'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
# gdf['Datetime'] = gdf['Datetime'].astype('datetime64[s, Pacific/Tongatapu]')
# Sort the DataFrame by time to ensure correct animation order
gdf = gdf.sort_values(by='Datetime')
gdf.drop(columns='Datetime', inplace=True)
gdf.head()

,DataId,CommId,DeviceName,DeviceDateTime,AgeInSeconds,BatteryVoltage,GpsQuality,Latitude,Longitude,SubmergedBoolean,Temperature0cm,Unnamed: 11,station,unixTime
386,66038667,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:45,63906,7.5,3,33.204360,-117.306144,0,-0.05,NaN,387,1772732700
385,66038669,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:50,64206,12.0,3,33.204357,-117.306149,0,NaN,NaN,386,1772733000
384,66038680,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:55,64507,12.0,3,33.204357,-117.306141,0,NaN,NaN,385,1772733300
383,66038795,3.010000e+14,USC/MTEL-UT-0001,3/5/26 18:00,64808,12.0,3,33.204363,-117.306149,0,NaN,NaN,384,1772733600
382,66038808,3.010000e+14,USC/MTEL-UT-0001,3/5/26 18:05,65108,12.0,3,33.204357,-117.306144,0,NaN,NaN,383,1772733900


## OpenOBS Data

In [13]:
for experiment in ["ISAC1", "ISAC2", "ISAC3"]:
    print(f".........................Assembling dataframe for {experiment}...")
    dfs = []
    if experiment == "ISAC1":
        filelist = ["ISAC1_top"]
        start = pd.Timestamp('2026-03-20 06:55', tz='Pacific/Tongatapu').timestamp()
        end = pd.Timestamp('2026-03-21 06:14', tz='Pacific/Tongatapu').timestamp()
    elif experiment == "ISAC2":
        filelist = ["ISAC2_mid_123"]
        start = pd.Timestamp('2026-03-21 12:00', tz='Pacific/Tongatapu').timestamp()
        end = pd.Timestamp('2026-03-23 11:31', tz='Pacific/Tongatapu').timestamp()
    elif experiment == "ISAC3":
        filelist = ["ISAC3_mid_113", "ISAC3_top_123", "ISAC3_mid_117"]
        start = pd.Timestamp('2026-03-22 14:13', tz='Pacific/Tongatapu').timestamp()
        end = pd.Timestamp('2026-03-23 12:38', tz='Pacific/Tongatapu').timestamp()

    for filename in filelist:
        df = pd.read_csv(filename+".TXT", skiprows=4)
        df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']

        # Change these lines to filter as you'd like:
        df['backscatter_filtered'] = df['backscatter'].rolling(window=20, center=True).median()
        df['ambient_filtered'] = df['ambient'].rolling(window=11, center=True).median()


        # 1. FIX THE "FAKE" UNIX TIME
        # Adding 8 hours (8 * 3600 seconds) to correct for the PST factory setting
        df['unixTime'] = df['unixTime'] + (8 * 3600)
        # Subtracting one hour because of DST:
        df['unixTime'] = df['unixTime'] - (1 * 3600)

        # 2. CONVERT TO LOCAL TONGA TIME
        df['Datetime'] = (
            pd.to_datetime(df['unixTime'], unit='s', utc=True)
            .dt.tz_convert('Pacific/Tongatapu')
        )

        # Now the sanity prints should make sense
        test_val = df.unixTime.iloc[0]
        print(f"--- Corrected Time Check  for {filename}.TXT ---")
        print(f"Start Time in UTC: {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(test_val))}")
        print(f"Local time in Tonga:   {df.Datetime.iloc[0]}")

        # Get S/N and rename columns
        with open(filename+".TXT", 'r') as file:
            lines = file.readlines()
            SN = lines[1].strip('\n') if len(lines) > 1 else "Unknown"

        suffix = "_" + SN[11:15]
        df.columns = [
            f"{col}{suffix}" if (col not in ['Datetime', 'unixTime']) else col
            for col in df.columns
        ]
        dfs.append(df)

    # Concatenate
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('unixTime') # Ensure global sort before merging GPS

    # 3. Solar position (Now uses correct local times)
    times = df['Datetime']
    solpos = get_solarposition(times, -21, -175)
    df['zenith'] = solpos['zenith'].values

    # 4. GPS Merge
    # Ensure gdf is sorted by unixTime too, or merge_asof will fail/error
    gdf = gdf.sort_values('unixTime')
    df = pd.merge_asof(df, gdf, on='unixTime', direction='backward')

    # Final Sanity Check
    peak_sun = df.loc[df['zenith'].idxmin()]
    print(f"\nSanity Check: Sun is highest (Zenith {peak_sun['zenith']:.2f}°) at: {peak_sun['Datetime']}")

    df.to_csv(experiment + '_full.csv', index=False)
    df = df[(df['unixTime'] >= start) & (df['unixTime'] <= end)]
    df.to_csv(experiment + '.csv', index=False)

.........................Assembling dataframe for ISAC1...
--- Corrected Time Check  for ISAC1_top.TXT ---
Start Time in UTC: 2026-03-19 16:32:55
Local time in Tonga:   2026-03-20 05:32:55+13:00

Sanity Check: Sun is highest (Zenith 106.63°) at: 2026-03-20 05:35:41+13:00
.........................Assembling dataframe for ISAC2...
--- Corrected Time Check  for ISAC2_mid_123.TXT ---
Start Time in UTC: 2026-03-20 22:42:29
Local time in Tonga:   2026-03-21 11:42:29+13:00

Sanity Check: Sun is highest (Zenith 21.15°) at: 2026-03-21 12:47:12+13:00
.........................Assembling dataframe for ISAC3...
--- Corrected Time Check  for ISAC3_mid_113.TXT ---
Start Time in UTC: 2026-03-22 00:42:35
Local time in Tonga:   2026-03-22 13:42:35+13:00
--- Corrected Time Check  for ISAC3_top_123.TXT ---
Start Time in UTC: 2026-03-21 23:31:16
Local time in Tonga:   2026-03-22 12:31:16+13:00
--- Corrected Time Check  for ISAC3_mid_117.TXT ---
Start Time in UTC: 2026-03-21 20:33:43
Local time in Tonga:   

## ISAC 3
